# 03 — Sistema de recomendación interpretativo


Objetivo: estimar la probabilidad de `>50K` y simular cambios en variables accionables para recomendar mejoras.
 Nota: recomendaciones basadas en correlaciones del dataset.


## 1) Carga de datos y preparación


In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/adult_clean.csv")  
X = df.drop("income", axis=1)
X.columns.tolist()[:5]


['age', 'workclass', 'fnlwgt', 'education', 'education.num']

## 2) Carga del modelo entrenado


In [3]:
import joblib
model = joblib.load("income_model.joblib")


## 3) Definición del perfil


In [4]:
perfil = {
    "age": 30,
    "workclass": "Private",
    "fnlwgt": 200000,
    "education": "HS-grad",
    "education-num": 9,
    "marital-status": "Never-married",
    "occupation": "Sales",
    "relationship": "Not-in-family",
    "race": "White",
    "sex": "Female",
    "capital-gain": 0,
    "capital-loss": 0,
    "hours-per-week": 30,
    "native-country": "United-States"
}


In [5]:
# Crear DataFrame del usuario con TODAS las columnas del entrenamiento
user_df = pd.DataFrame([perfil])

# Reordenar y asegurar columnas
user_df = user_df.reindex(columns=X.columns)


## 4) Normalización del perfil


## 5) Probabilidad base


In [6]:
import pandas as pd

# 1) Mapeo de nombres con guiones -> puntos (como espera el modelo)
rename_map = {
    "capital-gain": "capital.gain",
    "capital-loss": "capital.loss",
    "education-num": "education.num",
    "hours-per-week": "hours.per.week",
    "marital-status": "marital.status",
    "native-country": "native.country",
}

perfil_fix = {rename_map.get(k, k): v for k, v in perfil.items()}

# 2) DataFrame con el orden exacto de columnas
user_df = pd.DataFrame([perfil_fix]).reindex(columns=X.columns)

# 3) Rellenar valores faltantes (por seguridad)
for c in X.columns:
    if user_df[c].isna().any():
        if pd.api.types.is_numeric_dtype(X[c]):
            user_df[c] = user_df[c].fillna(X[c].median())
        else:
            user_df[c] = user_df[c].fillna(X[c].mode()[0])

# 4) Probabilidad base de >50K
classes = list(model.classes_)
idx = classes.index(">50K") if ">50K" in classes else 1

prob_base = model.predict_proba(user_df)[0, idx]
prob_base


np.float64(0.027533243806406548)

In [7]:
prob_base


np.float64(0.027533243806406548)

## 6) Recomendaciones (simulación)


In [8]:
def prob_gt50k(profile_dict):
    df1 = pd.DataFrame([profile_dict]).reindex(columns=X.columns)
    for c in X.columns:
        if df1[c].isna().any():
            if pd.api.types.is_numeric_dtype(X[c]):
                df1[c] = df1[c].fillna(X[c].median())
            else:
                df1[c] = df1[c].fillna(X[c].mode()[0])
    return model.predict_proba(df1)[0, idx]

base = prob_gt50k(perfil_fix)

opciones = {
    "education": sorted(X["education"].unique()),
    "occupation": sorted(X["occupation"].unique()),
    "hours.per.week": [30, 35, 40, 45, 50, 55, 60],
}

edu_num_map = df.groupby("education")["education.num"].median().to_dict()

sugs = []
for feat, vals in opciones.items():
    for v in vals:
        if perfil_fix.get(feat) == v:
            continue
        nuevo = perfil_fix.copy()
        nuevo[feat] = v
        if feat == "education":
            nuevo["education.num"] = float(edu_num_map.get(v, nuevo.get("education.num", 9)))
        p = prob_gt50k(nuevo)
        sugs.append((feat, v, p, p-base))

recs = (
    pd.DataFrame(sugs, columns=["feature","valor","prob_nueva","mejora"])
    .sort_values("mejora", ascending=False)
)

recs.head(10)


,feature,valor,prob_nueva,mejora
10,education,Doctorate,0.222874,0.195341
13,education,Prof-school,0.197100,0.169567
11,education,Masters,0.110898,0.083365
9,education,Bachelors,0.081732,0.054198
33,hours.per.week,60,0.073752,0.046219
32,hours.per.week,55,0.062810,0.035277
31,hours.per.week,50,0.053398,0.025865
7,education,Assoc-acdm,0.045891,0.018358
18,occupation,Exec-managerial,0.045492,0.017959
30,hours.per.week,45,0.045328,0.017795


## 7) Resumen listo para reportar


In [9]:
top5 = recs.head(5).copy()
top5["resumen"] = top5.apply(
    lambda r: f"{r['feature']} -> {r['valor']}: {base:.2%} → {r['prob_nueva']:.2%} (Δ {r['mejora']:.2%})",
    axis=1
)
top5["resumen"].tolist()


['education -> Doctorate: 2.75% → 22.29% (Δ 19.53%)',
 'education -> Prof-school: 2.75% → 19.71% (Δ 16.96%)',
 'education -> Masters: 2.75% → 11.09% (Δ 8.34%)',
 'education -> Bachelors: 2.75% → 8.17% (Δ 5.42%)',
 'hours.per.week -> 60: 2.75% → 7.38% (Δ 4.62%)']

Trabajar más o estudiar mejor?

Entrené un modelo de Machine Learning con datos del censo de EE.UU. para predecir si una persona superará los 50.000 USD anuales y desarrollé un sistema de recomendación interpretativo.

Para un perfil con una probabilidad inicial del 2.75%, el sistema muestra que aumentar el nivel educativo tiene un impacto mucho mayor que incrementar las horas de trabajo.

Por ejemplo, pasar de educación secundaria a un Doctorado aumenta la probabilidad hasta un 22.29%, mientras que trabajar 60 horas semanales solo la eleva hasta un 7.38%.

Estos resultados sugieren que, a largo plazo, invertir en educación es una estrategia más efectiva que el esfuerzo laboral extremo para mejorar los ingresos.

## 8) Interpretación breve (para la entrega)
- En este ejemplo, **educación** aporta un aumento de probabilidad mayor que aumentar solo las horas.
- Prioriza recomendaciones accionables: educación/formación, ocupación/sector, horas.
